# Решения: membership и счётчики

**Для преподавателя.** Полный эталон к `lesson.ipynb` и `homework.ipynb`; ученикам до сдачи не показывать.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def find_orders_csv():
    for path in (
        Path("orders_slim.csv"),
        Path("../orders_slim.csv"),
        Path("../../data/orders_slim.csv"),
        Path("../data/orders_slim.csv"),
        Path("../../../data/orders_slim.csv"),
    ):
        if path.exists():
            return path.resolve()
    return (
        "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/"
        "modules/08_08_logistics_clustering/data/orders_slim.csv"
    )


CSV_PATH = find_orders_csv()
DATE_COLUMNS = [
    "order_purchase_timestamp",
    "order_estimated_delivery_date",
    "order_delivered_customer_date",
]
df = pd.read_csv(CSV_PATH, parse_dates=DATE_COLUMNS)
assert len(df) > 0
assert df["order_id"].notna().all()
print(f"Загружено заказов: {len(df)}")


## Урок. 1–2. Membership и множества

In [ ]:
known = set(df["order_id"])
probes = df["order_id"].sample(min(8, len(df)), random_state=52).tolist() + ["missing"]
check = {oid: oid in known for oid in probes}
customer_states = set(df["customer_state"])
seller_states = set(df["seller_state"])
shared_states = customer_states & seller_states
assert check["missing"] is False


## Урок. 3–4. Счётчики пар

In [ ]:
pair_total, pair_late = {}, {}
for row in df[["seller_state", "customer_state", "is_late"]].itertuples(index=False):
    key = (row.seller_state, row.customer_state)
    pair_total[key] = pair_total.get(key, 0) + 1
    pair_late[key] = pair_late.get(key, 0) + int(row.is_late)
assert sum(pair_total.values()) == len(df)


## Урок. 5–7. Доли и фильтр

In [ ]:
pair_rate = {key: pair_late[key] / total for key, total in pair_total.items()}
MIN_SIZE = 3 if len(df) >= 30 else 1
eligible = {key: pair_rate[key] for key, total in pair_total.items() if total >= MIN_SIZE}
global_rate = float(df["is_late"].mean())
hot_segments = sorted([(key, rate, pair_total[key]) for key, rate in eligible.items() if rate > global_rate], key=lambda row: row[1], reverse=True)
assert all(total >= MIN_SIZE for _, _, total in hot_segments)


## Урок. 8. Watchlist

In [ ]:
hot_pairs = {pair for pair, _, _ in hot_segments}
watch = {row.order_id for row in df.itertuples() if (row.seller_state, row.customer_state) in hot_pairs and row.is_late}
assert watch <= set(df.loc[df["is_late"].eq(1), "order_id"])


## Урок. 9. Отчёт сегмента

In [ ]:
def segment_report(frame, seller_state, customer_state):
    part = frame[frame["seller_state"].eq(seller_state) & frame["customer_state"].eq(customer_state)]
    total, late = len(part), int(part["is_late"].sum())
    return {"total": total, "late": late, "rate": late / total if total else 0.0, "order_ids": part["order_id"].tolist()}

example_pair = next(iter(pair_total))
report = segment_report(df, *example_pair)
assert report["total"] == len(report["order_ids"])


## ДЗ. A1. Наблюдаемые продавцы

In [ ]:
watch_sellers = set(df.sort_values("delay_days", ascending=False)["seller_id"].head(10))
flags = [seller in watch_sellers for seller in df["seller_id"]]
assert len(flags) == len(df)


## ДЗ. A2–A3. Seller rate

In [ ]:
seller_total, seller_late = {}, {}
for row in df[["seller_state", "is_late"]].itertuples(index=False):
    seller_total[row.seller_state] = seller_total.get(row.seller_state, 0) + 1
    seller_late[row.seller_state] = seller_late.get(row.seller_state, 0) + int(row.is_late)
seller_rate = {s: seller_late[s] / seller_total[s] for s in seller_total}
median_total = float(np.median(list(seller_total.values())))
reliable = [(s, seller_total[s], seller_rate[s]) for s in seller_total if seller_total[s] >= median_total]
assert reliable


## ДЗ. Challenge

In [ ]:
cube = {}
for row in df[["seller_state", "customer_state", "is_late"]].itertuples(index=False):
    key = (row.seller_state, row.customer_state, int(row.is_late))
    cube[key] = cube.get(key, 0) + 1
SEGMENT_NOTE = "Сначала применяем минимум размера сегмента, затем сравниваем late rate с общей долей. Малый сегмент с rate 1.0 может быть случайностью. Решение — мониторить крупные устойчивые пары и не считать связь причиной задержки без дополнительных данных."
assert sum(cube.values()) == len(df) and len(SEGMENT_NOTE) >= 200
